# Schur-Weyl Duality Showcase

This notebook is designed as a compact research demo for showing:

1. A concrete three-photon state in Schur representation.
2. The block structure of passive linear-optical evolution in the same basis.
3. Invariant fingerprints before and after evolution.
4. A measurable output-pattern example in a three-mode tritter.

The first example follows the three-photon two-mode state discussed by Adamson et al. For `m_ext=2, n_particles=3`, the package now uses an explicit canonical symmetry-adapted basis, so entrywise comparison in `rho.density_matrix("schur")` is meaningful.


In [ ]:
import importlib.util
from pathlib import Path
import subprocess
import sys

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'photonic_jordan').is_dir():
        sys.path.insert(0, str(candidate))
        break

REPO_PIP = 'git+https://github.com/yangbc30/hierarchical-invariants-paper-code.git'
packages = []
if importlib.util.find_spec('photonic_jordan') is None:
    packages.append(REPO_PIP)
if importlib.util.find_spec('matplotlib') is None:
    packages.append('matplotlib')

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('photonic_jordan available:', importlib.util.find_spec('photonic_jordan') is not None)
print('using local repo path:', any((Path(p) / 'photonic_jordan').is_dir() for p in sys.path if Path(p).exists()))
print('matplotlib available:', importlib.util.find_spec('matplotlib') is not None)


In [ ]:
from itertools import permutations

import matplotlib.pyplot as plt
import numpy as np

from photonic_jordan import Fock, PhotonicSystem, safe_matmul

np.set_printoptions(precision=4, suppress=True, linewidth=180)


def accessible_density_from_symmetrized_products(visible_kets, hidden_kets):
    n = len(visible_kets)
    if n == 0 or len(hidden_kets) != n:
        raise ValueError('visible_kets and hidden_kets must be non-empty and have the same length.')

    m = int(np.asarray(visible_kets[0]).shape[0])
    d = int(np.asarray(hidden_kets[0]).shape[0])
    ext_dim = m ** n
    int_dim = d ** n

    psi = np.zeros(ext_dim * int_dim, dtype=complex)
    for perm in permutations(range(n)):
        ext_amp = np.asarray(visible_kets[perm[0]], dtype=complex)
        hid_amp = np.asarray(hidden_kets[perm[0]], dtype=complex)
        for k in range(1, n):
            ext_amp = np.kron(ext_amp, np.asarray(visible_kets[perm[k]], dtype=complex))
            hid_amp = np.kron(hid_amp, np.asarray(hidden_kets[perm[k]], dtype=complex))
        psi += np.kron(ext_amp, hid_amp)

    psi /= np.linalg.norm(psi)
    rho_tot = np.outer(psi, psi.conj()).reshape(ext_dim, int_dim, ext_dim, int_dim)
    rho_ext = np.trace(rho_tot, axis1=1, axis2=3)
    return rho_ext / np.trace(rho_ext)


def add_sector_boundary(ax, split=4):
    ax.axhline(split - 0.5, color='white', lw=2.0)
    ax.axvline(split - 0.5, color='white', lw=2.0)


def plot_real_imag(matrix, title, split=4):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    real = np.real(matrix)
    imag = np.imag(matrix)
    vmax = max(np.max(np.abs(real)), 1e-12)
    im0 = axes[0].imshow(real, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0].set_title(title + ' (real part)')
    add_sector_boundary(axes[0], split=split)
    fig.colorbar(im0, ax=axes[0], fraction=0.046)
    vmax = max(np.max(np.abs(imag)), 1e-12)
    im1 = axes[1].imshow(imag, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1].set_title(title + ' (imag part)')
    add_sector_boundary(axes[1], split=split)
    fig.colorbar(im1, ax=axes[1], fraction=0.046)
    for ax in axes:
        ax.set_xlabel('Schur basis index')
        ax.set_ylabel('Schur basis index')
    plt.tight_layout()
    return fig


def plot_abs(matrix, title, split=4):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(np.abs(matrix), cmap='viridis')
    ax.set_title(title)
    add_sector_boundary(ax, split=split)
    ax.set_xlabel('Schur basis index')
    ax.set_ylabel('Schur basis index')
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    return fig


def plot_distribution(prob_dict, title):
    labels = [str(k) for k in prob_dict.keys()]
    values = [prob_dict[k] for k in prob_dict.keys()]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(labels, values, color='#1f77b4')
    ax.set_title(title)
    ax.set_ylabel('probability')
    ax.set_xlabel('occupation pattern')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    return fig


def plot_invariant_bars(orders, before, after, title, ax):
    x = np.arange(len(orders))
    width = 0.38
    ax.bar(x - width / 2, before, width, label='before', color='#4c72b0')
    ax.bar(x + width / 2, after, width, label='after', color='#dd8452')
    ax.set_xticks(x)
    ax.set_xticklabels([str(j) for j in orders])
    ax.set_xlabel('order j')
    ax.set_ylabel('I_exact(j)')
    ax.set_title(title)
    ax.legend(frameon=False)


## 1) Adamson State in Canonical Schur Basis

We build the three-photon two-mode state corresponding to Eq. (27)/(30) in Adamson et al. The visible modes are the polarization basis states `H, V`; the hidden states are `a, a, c` with `c = (a+b)/sqrt(2)`.


In [ ]:
sys = PhotonicSystem(m_ext=2, n_particles=3, auto_cache=False)

H = np.array([1.0, 0.0], dtype=complex)
V = np.array([0.0, 1.0], dtype=complex)
a = np.array([1.0, 0.0], dtype=complex)
b = np.array([0.0, 1.0], dtype=complex)
c = (a + b) / np.sqrt(2.0)
omega = np.exp(2j * np.pi / 3.0)

rho_ext = accessible_density_from_symmetrized_products(
    visible_kets=[H + V, H + omega * V, H + (omega ** 2) * V],
    hidden_kets=[a, a, c],
)
rho = sys.state.from_density_matrix(rho_ext, label='Adamson Eq. (30)')
rho_s = rho.density_matrix(rep='schur')

print('Sector weights:')
print(rho.sector_weights())
print('\nOutput pattern distribution:')
print(rho.pattern_distribution())
print('\nSchur density matrix:')
print(rho_s)

plot_real_imag(rho_s, 'Adamson accessible state in Schur basis', split=4)
plt.show()


## 2) Evolution in Schur Basis

Apply a balanced two-mode beamsplitter. In Schur basis, both the state and the unitary exhibit the same sector block structure. In the `(2,1)` sector, the multiplicity structure appears as two equivalent `2x2` blocks; depending on basis convention they may differ by signs or phases, but their absolute-value pattern is the same.


In [ ]:
S = np.array([[1.0, 1.0], [1.0, -1.0]], dtype=complex) / np.sqrt(2.0)
U_tensor = sys.space.total_unitary_from_single_particle(S)
W = sys.decomposition.schur_transform()
U_s = safe_matmul(W.conj().T, U_tensor, W)

rho_out = rho.evolve(S)
rho_out_s = rho_out.density_matrix(rep='schur')

print('Mixed-sector 2x2 blocks of U_s:')
print(U_s[4:6, 4:6])
print(U_s[6:8, 6:8])
print('Absolute values equal:', np.allclose(np.abs(U_s[4:6, 4:6]), np.abs(U_s[6:8, 6:8])))

plot_abs(U_s, 'Absolute value of U in Schur basis', split=4)
plt.show()
plot_real_imag(rho_out_s, 'Evolved Adamson state in Schur basis', split=4)
plt.show()


## 3) Invariant Fingerprints

Jordan invariants should be unchanged by passive linear-optical evolution. We compare the exact-layer fingerprints before and after the beamsplitter, both globally and within the nontrivial `(2,1)` Schur sector.


In [ ]:
orders = list(range(4))
global_before = [rho.invariant.I_exact(j) for j in orders]
global_after = [rho_out.invariant.I_exact(j) for j in orders]
sector_before = [rho.invariant.I_exact(j, sector=(2, 1)) for j in orders]
sector_after = [rho_out.invariant.I_exact(j, sector=(2, 1)) for j in orders]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
plot_invariant_bars(orders, global_before, global_after, 'Global exact-layer invariants', axes[0])
plot_invariant_bars(orders, sector_before, sector_after, 'Sector (2,1) exact-layer invariants', axes[1])
plt.tight_layout()
plt.show()

print('Global invariants before:', global_before)
print('Global invariants after :', global_after)
print('Sector (2,1) before    :', sector_before)
print('Sector (2,1) after     :', sector_after)


## 4) Output Patterns: Three-Photon Tritter Suppression

As a directly measurable many-body effect, we evolve the bosonic Fock input `|1,1,1>` through a balanced tritter. The `(2,1,0)`-type outputs are suppressed, while only `(3,0,0)` and `(1,1,1)`-type patterns remain.


In [ ]:
U_tritter = np.array(
    [
        [1.0, 1.0, 1.0],
        [1.0, omega, omega ** 2],
        [1.0, omega ** 2, omega],
    ],
    dtype=complex,
) / np.sqrt(3.0)

rho_in = Fock(1, 1, 1, auto_cache=False)
rho_tritter = rho_in.evolve(U_tritter)
dist = rho_tritter.pattern_distribution()

print('Tritter output pattern distribution:')
for occ, p in dist.items():
    print(f'{occ}: {p:.6f}')

plot_distribution(dist, 'Three-photon output patterns after a balanced tritter')
plt.show()
